# Bab 3 - Pengenalan Prediksi Saraf: Forward Propagation

**Buku:** *Grokking Deep Learning* (Andrew W. Trask, Manning)

## Ringkasan bab
Bab ini hanya mencakup langkah **predict (prediksi)** dari alur predict-compare-learn, yang disebut **forward propagation**: bagaimana jaringan mengubah input menjadi prediksi. Topik:

1. Jaringan saraf dengan **satu input dan satu bobot** (bobot hanya mengalikan input).
2. **Banyak input** -> **penjumlahan berbobot (weighted sum)** (dot product).
3. **Banyak output** dari satu input.
4. **Banyak input dan banyak output** -> *matriks* bobot.
5. **Hidden layer** -> menumpuk jaringan (output dari satu layer menjadi input untuk layer berikutnya).
6. Pengenalan singkat tentang **numpy** untuk operasi vektor/matriks.

## Teori secara mendalam
* **Bobot** menyatakan seberapa besar input tertentu mempengaruhi prediksi. "Pengetahuan" jaringan saraf disimpan dalam bobotnya.
* **Input** adalah data yang kita ketahui *pada saat prediksi*; **prediksi** adalah apa yang ingin kita ketahui.
* **Penjumlahan berbobot / dot product:** untuk input `x` dan bobot `w`, `pred = sum_i x_i * w_i`. Secara intuitif, prediksi bernilai tinggi ketika input dengan bobot besar sedang "aktif". Ini juga merupakan ukuran **kemiripan** antara vektor input dan vektor bobot.
* **Banyak output:** dengan satu input dan beberapa output, setiap output memiliki bobotnya sendiri; input yang sama diskalakan oleh bobot yang berbeda.
* **Bentuk matriks:** untuk `n` input dan `m` output, bobot membentuk matriks `m x n` `W` dan `pred = W @ x` (sebuah *perkalian vektor-matriks*).
* **Hidden layer:** `hidden = W1 @ x`, `pred = W2 @ hidden`. Vektor perantara disebut layer *tersembunyi (hidden)* karena tidak diamati secara langsung dalam data. Menumpuk layer menghasilkan jaringan yang *dalam (deep)*. (Tanpa fungsi non-linearitas, yang baru diperkenalkan pada Bab 6, dua layer linear yang ditumpuk akan runtuh menjadi satu pemetaan linear saja.)

In [1]:
import numpy as np

## 1. Satu input, satu bobot
Jaringan dengan input tunggal (jumlah rata-rata jari kaki per pemain, contoh yang menyenangkan) dan bobot tunggal yang menghasilkan satu prediksi.

In [2]:
def neural_network(inp, weight):
    return inp * weight

weight = 0.1
number_of_toes = [8.5, 9.5, 10, 9]
inp = number_of_toes[0]
print('prediksi:', neural_network(inp, weight))

prediction: 0.8500000000000001


## 2. Banyak input -> penjumlahan berbobot
Input: rata-rata jari kaki, rekor menang/kalah, jumlah penggemar (dalam jutaan). Bobot menentukan seberapa penting masing-masing variabel.

In [3]:
def w_sum(a, b):
    assert len(a) == len(b)
    return sum(a[i] * b[i] for i in range(len(a)))

weights = [0.1, 0.2, 0.0]

def neural_network(inp, weights):
    return w_sum(inp, weights)

toes   = [8.5, 9.5, 9.9, 9.0]
wlrec  = [0.65, 0.8, 0.8, 0.9]
nfans  = [1.2, 1.3, 0.5, 1.0]

inp = [toes[0], wlrec[0], nfans[0]]
print('prediksi (python murni):', neural_network(inp, weights))

# numpy melakukan hal yang sama dengan satu dot product
print('prediksi (numpy):       ', np.array(inp).dot(np.array(weights)))

prediction (pure python): 0.9800000000000001
prediction (numpy):       0.9800000000000001


## 3. Satu input, banyak output
Input yang sama dikalikan dengan satu bobot per output (di sini: *cedera/hurt*, *menang/win*, *sedih/sad*).

In [4]:
def ele_mul(number, vector):
    return [number * v for v in vector]

weights = [0.3, 0.2, 0.9]

def neural_network(inp, weights):
    return ele_mul(inp, weights)

wlrec = [0.65, 1.0, 1.0, 0.9]
print('prediksi hurt, win, sad:', neural_network(wlrec[0], weights))

hurt, win, sad predictions: [0.195, 0.13, 0.5850000000000001]


## 4. Banyak input dan banyak output
Setiap output memiliki baris bobotnya sendiri; ini adalah perkalian vektor-matriks.

In [5]:
#            toes  %win  #fans
weights = [[0.1, 0.1, -0.3],   # hurt?
           [0.1, 0.2,  0.0],   # win?
           [0.0, 1.3,  0.1]]   # sad?

def vect_mat_mul(vect, matrix):
    return [w_sum(vect, row) for row in matrix]

def neural_network(inp, weights):
    return vect_mat_mul(inp, weights)

inp = [toes[0], wlrec[0], nfans[0]]
print('python murni:', neural_network(inp, weights))
print('numpy:       ', np.array(weights) @ np.array(inp))

pure python: [0.5550000000000002, 0.9800000000000001, 0.9650000000000001]
numpy:       [0.555 0.98  0.965]


## 5. Hidden layer: menumpuk jaringan
Output dari matriks bobot pertama (hidden layer) menjadi input untuk matriks bobot kedua.

In [6]:
ih_wgt = np.array([[0.1, 0.2, -0.1],    # hid[0]
                   [-0.1, 0.1, 0.9],    # hid[1]
                   [0.1, 0.4, 0.1]]).T  # hid[2]   -> bentuk (3 input, 3 hidden)

hp_wgt = np.array([[0.3, 1.1, -0.3],    # hurt?
                   [0.1, 0.2, 0.0],     # win?
                   [0.0, 1.3, 0.1]]).T  # sad?     -> bentuk (3 hidden, 3 output)

def neural_network(inp, weights):
    hid = inp.dot(weights[0])
    pred = hid.dot(weights[1])
    return pred

inp = np.array([toes[0], wlrec[0], nfans[0]])
print('prediksi melalui hidden layer:', neural_network(inp, [ih_wgt, hp_wgt]))

prediction through the hidden layer: [0.2135 0.145  0.5065]


### Mengapa hidden layer saja belum menambah kekuatan (saat ini)
Dua layer *linear* yang ditumpuk sama dengan satu layer linear yang matriksnya adalah hasil kali dari keduanya. Aktivasi non-linear (Bab 6 dan 9) adalah hal yang membuat kedalaman memberikan kekuatan.

In [7]:
combined = ih_wgt @ hp_wgt
print('hasil sama dengan SATU matriks:', inp @ combined)

same result with ONE matrix: [0.2135 0.145  0.5065]


## 6. Beberapa fakta numpy yang digunakan di seluruh buku

In [8]:
a = np.array([0, 1, 2, 3]); b = np.array([4, 5, 6, 7])
print('perkalian elemen demi elemen:', a * b)
print('penjumlahan elemen demi elemen:', a + b)
print('perkalian skalar:           ', a * 0.5)
print('dot product:                ', a.dot(b))
m = np.random.rand(3, 2)
print('bentuk matriks:', m.shape, ' bentuk hasil vektor-matriks:', np.random.rand(3).dot(m).shape)

elementwise multiply: [ 0  5 12 21]
elementwise add:      [ 4  6  8 10]
scalar multiply:      [0.  0.5 1.  1.5]
dot product:          38
matrix shape: (3, 2)  vector-matrix result shape: (2,)


## Poin-poin kunci
* Forward propagation = mengubah input menjadi prediksi menggunakan bobot.
* Penjumlahan berbobot adalah operasi inti; dengan banyak output, operasi ini menjadi perkalian matriks.
* Hidden layer menyalurkan satu matriks bobot ke matriks berikutnya, yang merupakan cara jaringan "dalam (deep)" dibuat.
* Prediksi saja belum mempelajari apa pun: Bab 4 menambahkan langkah membandingkan (compare) dan belajar (learn).